# PI-CAI nnUNet v2 — Domain-Adaptive Training Pipeline

**Architecture**: 3D Full-Resolution U-Net with 6-Channel Input
- T2W + ADC + HBV + 3 One-Hot Zonal Mask Channels
- Custom Adaptive Focal Loss (Dice + Dynamic α/γ Focal)
- Domain Adaptation: Train on RUMC+ZGT, Holdout PCNN

## Workflow
| Section | Purpose |
|---|---|
| **Setup** | Mount Drive, install nnUNet, set env vars |
| **Data Restore** | Unzip preprocessed cache from Drive (~5 min) |
| **Pre-Flight Checks** | Strict verification before burning GPU hours |
| **Training** | 1000 epochs, Fold 0, Adaptive Focal Loss |

---

# ═══════════════════════════════════════════════════
# SECTION 1: SETUP
# ═══════════════════════════════════════════════════

## 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Verify source dataset exists
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'
assert os.path.exists(SOURCE_DATA), f'Dataset not found at {SOURCE_DATA}'

for folder in ['t2', 'adc_reg', 'hbv_reg', 'zonal_masks', 'lesion_masks']:
    path = os.path.join(SOURCE_DATA, folder)
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.nii.gz')])
        print(f'  {folder}: {count} files')
    else:
        print(f'  ⚠️ {folder}: NOT FOUND')

print(f'\n✅ Dataset verified at {SOURCE_DATA}')

## 1.2 Clone Repository & Install nnUNet

In [ ]:
import os

REPO_DIR = '/content/nnu-Net-Baseline'

if os.path.exists(REPO_DIR):
    print('Repository already cloned, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/HemishJain09/nnu-Net-Baseline.git {REPO_DIR}

print(f'\n✅ Repository ready')
!ls {REPO_DIR}/*.py

In [ ]:
# Clone nnUNet from source and install in editable mode
!rm -rf /content/nnUNet
!git clone https://github.com/MIC-DKFZ/nnUNet.git /content/nnUNet
!cd /content/nnUNet && pip install -e .
!pip install -q nibabel scikit-learn tqdm SimpleITK

# Verify installation
!nnUNetv2_plan_and_preprocess --help | head -3
print('\n✅ nnUNet v2 installed from source successfully')

## 1.3 Set Environment Variables

**I/O Architecture:**
- `nnUNet_raw` + `nnUNet_preprocessed` → Local NVMe SSD (fast I/O for GPU)
- `nnUNet_results` → Google Drive (persistent checkpoints)

In [ ]:
import os

# Fast local SSD (ephemeral — wiped on disconnect)
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'

# Persistent Google Drive (checkpoints survive disconnects)
RESULTS_DIR = '/content/drive/MyDrive/PI-CAI_nnUNet_Results'
os.environ['nnUNet_results'] = RESULTS_DIR

# Create all directories
for d in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Environment Variables:')
print(f'  nnUNet_raw:          {os.environ["nnUNet_raw"]}          (local SSD)')
print(f'  nnUNet_preprocessed: {os.environ["nnUNet_preprocessed"]} (local SSD)')
print(f'  nnUNet_results:      {os.environ["nnUNet_results"]}      (Google Drive)')

!echo '' && df -h /content | tail -1 | awk '{print "Local SSD: " $4 " free of " $2}'
print('\n✅ Environment configured')

# ═══════════════════════════════════════════════════
# SECTION 2: DATA RESTORE (from Zip Cache)
# ═══════════════════════════════════════════════════

Restores your 4-hour preprocessed dataset from Google Drive in ~5 minutes.
If the cache doesn't exist, falls back to full conversion + preprocessing.

In [ ]:
import os, shutil

CACHE_ZIP = '/content/drive/MyDrive/PI-CAI_nnUNet_Preprocessed_Cache.zip'
PREPROCESSED_DIR = os.environ['nnUNet_preprocessed']
RAW_DIR = os.environ['nnUNet_raw']
REPO_DIR = '/content/nnu-Net-Baseline'
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'

if os.path.exists(CACHE_ZIP):
    print(f"📦 Cache HIT: Found {CACHE_ZIP}!")
    print("Unzipping directly to local SSD (~5-10 minutes)...")

    if os.path.exists(PREPROCESSED_DIR):
        shutil.rmtree(PREPROCESSED_DIR)
    os.makedirs(PREPROCESSED_DIR)

    !unzip -q -o {CACHE_ZIP} -d {PREPROCESSED_DIR}
    print("✅ Successfully restored preprocessed data from Cache!")
else:
    print(f"❌ Cache MISS: No zip found at {CACHE_ZIP}.")
    print("Running full 1500-patient Conversion and Preprocessing (~4 hours)...")

    print("\n1/3: Converting raw PI-CAI data to nnUNet format...")
    !python {REPO_DIR}/convert_picai_to_nnunet.py \\
        --source_dir {SOURCE_DATA} \\
        --nnunet_raw {RAW_DIR}

    print("\n2/3: Running nnUNetv2_plan_and_preprocess...")
    !nnUNetv2_plan_and_preprocess -d 500 --verify_dataset_integrity -c 3d_fullres

    print("\n3/3: Creating Zip Cache on Google Drive...")
    import shutil as sh
    sh.make_archive(CACHE_ZIP.replace('.zip', ''), 'zip', PREPROCESSED_DIR)
    print(f"✅ Preprocessed data cached to {CACHE_ZIP}!")

### 2.1 Restore Raw Data (needed for splits generation)

The Zip Cache only contains `nnUNet_preprocessed`. We also need the `labelsTr` folder
in `nnUNet_raw` so that `generate_splits.py` can enumerate all case IDs.

In [ ]:
import os

RAW_DIR = os.environ['nnUNet_raw']
REPO_DIR = '/content/nnu-Net-Baseline'
SOURCE_DATA = '/content/drive/MyDrive/PI-CAI_pre-processed'

labels_dir = os.path.join(RAW_DIR, 'Dataset500_PICAI', 'labelsTr')

if os.path.exists(labels_dir) and len(os.listdir(labels_dir)) > 0:
    print(f'✅ labelsTr already exists with {len(os.listdir(labels_dir))} files. Skipping.')
else:
    print('labelsTr not found in nnUNet_raw. Rebuilding from source...')
    # We only need the raw conversion for the label files.
    # Since the cache restored preprocessed data, we run a quick conversion.
    !python {REPO_DIR}/convert_picai_to_nnunet.py \\
        --source_dir {SOURCE_DATA} \\
        --nnunet_raw {RAW_DIR}
    print(f'✅ Raw data restored.')

# ═══════════════════════════════════════════════════
# SECTION 3: PRE-FLIGHT CHECKS (Strict Verification)
# ═══════════════════════════════════════════════════

**⚠️ DO NOT SKIP THIS SECTION.**

These checks verify every single component of the pipeline before
you burn 10+ hours of GPU time. If any check fails, training WILL crash.

In [ ]:
import os, json, sys
import numpy as np

REPO_DIR = '/content/nnu-Net-Baseline'
PREPROCESSED_DIR = os.environ['nnUNet_preprocessed']
RAW_DIR = os.environ['nnUNet_raw']
RESULTS_DIR = os.environ['nnUNet_results']

errors = []
warnings = []

print('🔍 PRE-FLIGHT CHECK: Starting strict verification...\n')
print('=' * 60)

# ─── CHECK 1: Preprocessed Data Exists ───────────────────
print('\n[1/8] Checking preprocessed data...')
plans_path = os.path.join(PREPROCESSED_DIR, 'Dataset500_PICAI', 'nnUNetPlans.json')
if os.path.exists(plans_path):
    with open(plans_path) as f:
        plans = json.load(f)
    configs = list(plans.get('configurations', {}).keys())
    print(f'  ✅ nnUNetPlans.json found. Configs: {configs}')
else:
    errors.append('nnUNetPlans.json not found! Cache may be corrupted.')
    print(f'  ❌ MISSING: {plans_path}')

# ─── CHECK 2: Preprocessed .npz files ─────────────────────
print('\n[2/8] Counting preprocessed patient files...')
npz_dir = os.path.join(PREPROCESSED_DIR, 'Dataset500_PICAI', 'nnUNetPlans_3d_fullres')
if os.path.exists(npz_dir):
    npz_files = [f for f in os.listdir(npz_dir) if f.endswith('.npz')]
    print(f'  ✅ Found {len(npz_files)} preprocessed .npz files')
    if len(npz_files) < 1400:
        errors.append(f'Expected ~1500 .npz files, found only {len(npz_files)}. Cache is incomplete!')
else:
    errors.append(f'3d_fullres preprocessed directory not found at {npz_dir}')
    print(f'  ❌ MISSING: {npz_dir}')

# ─── CHECK 3: Raw labelsTr exists ─────────────────────────
print('\n[3/8] Checking raw labelsTr directory...')
labels_dir = os.path.join(RAW_DIR, 'Dataset500_PICAI', 'labelsTr')
if os.path.exists(labels_dir):
    label_count = len([f for f in os.listdir(labels_dir) if f.endswith('.nii.gz')])
    print(f'  ✅ labelsTr has {label_count} files')
    if label_count < 1400:
        errors.append(f'Expected ~1500 labels, found only {label_count}')
else:
    errors.append('labelsTr directory not found! Cannot generate splits.')
    print(f'  ❌ MISSING: {labels_dir}')

# ─── CHECK 4: marksheet.csv ────────────────────────────────
print('\n[4/8] Checking marksheet.csv...')
if os.path.exists('/content/marksheet.csv'):
    import pandas as pd
    df = pd.read_csv('/content/marksheet.csv')
    centers = df['center'].unique().tolist()
    print(f'  ✅ marksheet.csv loaded. {len(df)} rows, Centers: {centers}')
else:
    errors.append('marksheet.csv not found at /content/marksheet.csv!')
    print('  ❌ MISSING: /content/marksheet.csv')

# ─── CHECK 5: Focal Loss Injection ─────────────────────────
print('\n[5/8] Testing Focal Loss injection...')
inject_path = os.path.join(REPO_DIR, 'inject_focal_loss.py')
if os.path.exists(inject_path):
    print(f'  ✅ inject_focal_loss.py exists')
else:
    errors.append('inject_focal_loss.py not found in repository!')
    print(f'  ❌ MISSING: {inject_path}')

# ─── CHECK 6: Focal Loss Math Verification ─────────────────
print('\n[6/8] Running Focal Loss mathematical verification...')
import torch
from torch import nn
try:
    # Inline the AdaptiveFocalLoss to test it right here
    class _TestFocalLoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.ce = nn.CrossEntropyLoss(reduction='none')
        def forward(self, x, t):
            ce = self.ce(x, t)
            pt = torch.exp(-ce)
            return (((1-pt)**2.0) * ce).mean()

    _fl = _TestFocalLoss()
    _x = torch.randn(2, 2, 4, 4, 4, requires_grad=True)
    _t = torch.randint(0, 2, (2, 4, 4, 4))
    _loss = _fl(_x, _t.long())
    _loss.backward()
    assert _x.grad is not None and not torch.isnan(_x.grad).any()
    print(f'  ✅ Forward + Backward pass OK. Loss={_loss.item():.4f}')
    del _fl, _x, _t, _loss
except Exception as e:
    errors.append(f'Focal Loss math test failed: {e}')
    print(f'  ❌ FAILED: {e}')

# ─── CHECK 7: GPU Available ────────────────────────────────
print('\n[7/8] Checking GPU...')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'  ✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
    if gpu_mem < 10:
        warnings.append(f'GPU has only {gpu_mem:.1f} GB VRAM. Consider switching to A100 for 3d_fullres.')
else:
    errors.append('No GPU detected! Go to Runtime > Change runtime type > GPU')
    print('  ❌ NO GPU!')

# ─── CHECK 8: Disk Space ───────────────────────────────────
print('\n[8/8] Checking disk space...')
stat = os.statvfs('/content')
free_gb = (stat.f_bavail * stat.f_frsize) / (1024**3)
print(f'  Local SSD: {free_gb:.1f} GB free')
if free_gb < 20:
    warnings.append(f'Only {free_gb:.1f} GB free on local SSD. Training may fail due to disk space.')

# ─── FINAL VERDICT ─────────────────────────────────────────
print('\n' + '=' * 60)
if errors:
    print('❌ PRE-FLIGHT CHECK FAILED!')
    print(f'   {len(errors)} critical error(s):')
    for e in errors:
        print(f'   • {e}')
    print('\n🛑 DO NOT PROCEED TO TRAINING. Fix the errors above first.')
    raise RuntimeError(f'Pre-flight failed with {len(errors)} error(s)')
else:
    print('✅ ALL PRE-FLIGHT CHECKS PASSED!')
    if warnings:
        print(f'   ⚠️ {len(warnings)} warning(s):')
        for w in warnings:
            print(f'   • {w}')
    print('\n🚀 You are cleared for training.')
print('=' * 60)

# ═══════════════════════════════════════════════════
# SECTION 4: GENERATE SPLITS & INJECT FOCAL LOSS
# ═══════════════════════════════════════════════════

## 4.1 Generate Domain Adaptation Splits (Exclude PCNN)

In [ ]:
REPO_DIR = '/content/nnu-Net-Baseline'

!python {REPO_DIR}/generate_splits.py \\
    --nnunet_raw $nnUNet_raw  \\
    --nnunet_preprocessed $nnUNet_preprocessed  \\
    --marksheet /content/marksheet.csv \\
    --train_centers RUMC ZGT

## 4.2 Inject Adaptive Focal Loss & Set 1000 Epochs

In [ ]:
REPO_DIR = '/content/nnu-Net-Baseline'

# Inject custom Adaptive Focal Loss trainer into nnUNet source
!python {REPO_DIR}/inject_focal_loss.py

# Force 1000 epochs (handles any previous value)
trainer_file = '/content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py'
import re
with open(trainer_file, 'r') as f:
    content = f.read()
content = re.sub(r'self\.num_epochs = \d+', 'self.num_epochs = 1000', content)
with open(trainer_file, 'w') as f:
    f.write(content)

# Verify
!grep 'self.num_epochs' {trainer_file}
print('\n✅ Focal Loss injected. Epochs set to 1000.')

# ═══════════════════════════════════════════════════
# SECTION 5: TRAINING — Fold 0 (1000 Epochs)
# ═══════════════════════════════════════════════════

**Expected Duration**: ~10-15 hours on A100, ~20+ hours on T4

**Auto-Resume**: nnUNet saves checkpoints to Google Drive every 50 epochs.
If Colab disconnects:
1. Re-run all cells from Section 1 (they are idempotent)
2. The cache will restore data in 5 min
3. This cell will detect the existing checkpoint and resume

In [ ]:
# Train Fold 0 with Adaptive Focal Loss
!nnUNetv2_train 500 3d_fullres 0 -tr nnUNetTrainerFocalLoss

# ═══════════════════════════════════════════════════
# SECTION 6: OPTIONAL — Additional Folds & Inference
# ═══════════════════════════════════════════════════

In [ ]:
# Uncomment one fold at a time:
# !nnUNetv2_train 500 3d_fullres 1 -tr nnUNetTrainerFocalLoss
# !nnUNetv2_train 500 3d_fullres 2 -tr nnUNetTrainerFocalLoss
# !nnUNetv2_train 500 3d_fullres 3 -tr nnUNetTrainerFocalLoss
# !nnUNetv2_train 500 3d_fullres 4 -tr nnUNetTrainerFocalLoss

In [ ]:
# After training, run inference on holdout center (PCNN):
# !nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 500 -c 3d_fullres -f 0 -tr nnUNetTrainerFocalLoss